In [275]:
from typing import TypedDict, List
from random import randint
from langgraph.graph import StateGraph, START, END

In [276]:
class AgentState(TypedDict):
    player_name: str
    guesses: List[int]
    attempts: int
    lower_bound: int
    upper_bound: int
    answer: int
    hint: List[str]
    result: str

In [277]:
def setup(state: AgentState) -> AgentState:
    """This function sets up the game."""
    
    upper = state['upper_bound']
    lower = state['lower_bound']
    
    state['answer'] = randint(lower, upper)
    
    state['attempts'] = 0
    return state

In [278]:
def guess(state: AgentState) -> AgentState:
    """This function guesses the number between the upper and lower bound."""
    upper = state['upper_bound']
    lower = state['lower_bound']
    
    if not state['hint']:
        guess = int((lower + upper) / 2)
        state['guesses'].append(guess)
    
    else:
        last_guess = state['guesses'][-1]
        latest_hint = state['hint'][-1]
        
        if latest_hint == "higher":
            state['lower_bound'] = last_guess
            guess = int((last_guess + upper) / 2)
            state['guesses'].append(guess)
        else:
            state['upper_bound'] = last_guess
            guess = int((lower + last_guess) / 2)
            state['guesses'].append(guess)
        
    state['attempts'] += 1
    
    return state

In [279]:
def hint_node(state: AgentState) -> AgentState:
    """This node gives hint for the player's guess as 'higher' or 'lower'."""
    latest_guess = state['guesses'][-1]
    answer = state['answer']
    
    if state["attempts"] >= 7:
        state["result"] = f"Sorry {state['player_name']}, you have failed to guess the correct answer within the given number of attempts. Better Luck next time!"
    
    if latest_guess == answer:
        state["result"] = f"Congratulations {state['player_name']}, you have guessed the correct answer in just {state['attempts']} attempts.!"
        
    elif latest_guess < answer:
        state['hint'].append("higher")

    else:
        state['hint'].append("lower")
        
    return state

In [280]:
def should_continue(state: AgentState):
    """This is a function for conditionally routing to either back to guess node or exit."""
    latest_guess = 0
    answer = state['answer']
    
    if state['guesses']:
        latest_guess = state['guesses'][-1]
        
    if state['attempts'] >= 7 or latest_guess == answer:
        return "exit"
    else:
        return "continue"
    

In [281]:
graph = StateGraph(AgentState)

graph.add_node("setup", setup)
graph.add_node("guess", guess)
graph.add_node("hint", hint_node)

graph.add_edge(START, "setup")
graph.add_edge("setup", "guess")
graph.add_edge("guess", "hint")

graph.add_conditional_edges(
    "hint",
    should_continue,
    
    {
        "continue": "guess",
        "exit": END
    }
)

app = graph.compile()

In [282]:
app.invoke({
    "player_name": "Hari",
    "guesses": [],
    "attempts": 7,
    "lower_bound": 1,
    "upper_bound": 20,
    "hint": [],
    "result": ''
})

{'player_name': 'Hari',
 'guesses': [10, 5, 7, 8, 9],
 'attempts': 5,
 'lower_bound': 8,
 'upper_bound': 10,
 'answer': 9,
 'hint': ['lower', 'higher', 'higher', 'higher'],
 'result': 'Congratulations Hari, you have guessed the correct answer in just 5 attempts.!'}